## Surface Code Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymatching
import stim
from surface_unrotated_d3 import build_surface_code as build_unrotated
from surface_rotated_d3 import build_surface_code as build_rotated


In [ ]:
# params
physical_error = np.logspace(-4,-1,10) # 10^-4 -> 10^-1
rounds_list = [3,5,10]
shots = 10000

results_unrotated = {r: [] for r in rounds_list}
results_rotated = {r: [] for r in rounds_list}

print("Starting Side-by-Side Simulation!")

for r in rounds_list:
    print(f"Debugging: Simulating {r} rounds.")
    for p in physical_error:
        # Unrotated Simulation
        c_un = build_unrotated(rounds=r, p=p)
        err_mod_un = c_un.detector_error_model(decompose_errors=True)
        match_un = pymatching.Matching.from_detector_error_model(err_mod_un)
        samp_un = c_un.compile_detector_sampler()
        syn_un, obs_un = samp_un.sample(shots=shots, separate_observables=True)
        pred_un = match_un.decode_batch(syn_un)
        log_err_un = np.sum(np.any(pred_un != obs_un, axis=1)) / shots
        results_unrotated[r].append(log_err_un)
        
        # Rotated Simulation
        c_rot = build_rotated(rounds=r, p=p)
        err_mod_rot = c_rot.detector_error_model(decompose_errors=True)
        match_rot = pymatching.Matching.from_detector_error_model(err_mod_rot)
        samp_rot = c_rot.compile_detector_sampler()
        syn_rot, obs_rot = samp_rot.sample(shots=shots, separate_observables=True)
        pred_rot = match_rot.decode_batch(syn_rot)
        log_err_rot = np.sum(np.any(pred_rot != obs_rot, axis=1)) / shots
        results_rotated[r].append(log_err_rot)
        
print("Execution Complete!")


In [ ]:
plt.figure(figsize=(12,6))

colors = {3: 'blue', 5: 'orange', 10: 'green'}

for r in rounds_list:
    # Plot Unrotated (Dashed)
    plt.plot(physical_error, results_unrotated[r], '--', marker='x', color=colors[r], label=f'{r} Rounds (Unrotated 25q)')
    # Plot Rotated (Solid)
    plt.plot(physical_error, results_rotated[r], '-', marker='o', color=colors[r], label=f'{r} Rounds (Rotated 17q)')

plt.xscale('log')
plt.yscale('log')
plt.axline((1e-4, 1e-4), (1e-1, 1e-1), color='black', linestyle=':', alpha=0.5, label='Logical Error = Physical Error')

plt.xlabel('Physical Error Rate')
plt.ylabel('Logical Error Rate')
plt.title('Surface Code Thresholds: Unrotated vs Rotated ($d=3$)')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, which="both", ls='-', alpha=0.1)
plt.tight_layout()
plt.show()
